In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install transformers==4.40.1 peft==0.10.0 accelerate==0.27.2 -q


# 🚨 NLP with Disaster Tweets – Exploratory Data Analysis (EDA)


## 🧠 Overview

This notebook explores the dataset for the Kaggle competition: [Natural Language Processing with Disaster Tweets](https://www.kaggle.com/competitions/nlp-getting-started).

🧨 **Goal**: Predict which tweets are about real disasters and which are not.  
📊 **Objective**: Perform an in-depth EDA to better understand the structure, patterns, and quality of the data before modeling.


In [ ]:
import pandas as pd

train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
sample_submission = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')

train.head()


## 📦 Load Data and Preview


In [ ]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)

train.info()
train.describe()


In [ ]:
train.isnull().sum()


## 🚫 Missing Values
We visualize where the dataset has missing values using a heatmap.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(train.isnull(), cbar=False)
plt.title("Missing Values in Train Set")
plt.show()

(train.isnull().sum() / len(train)).sort_values(ascending=False)


## 🎯 Class Balance
How balanced is our target variable?


In [ ]:
sns.countplot(x='target', data=train)
plt.title("Distribution of Disaster vs Non-Disaster Tweets")
plt.xticks([0, 1], ['Not Disaster', 'Disaster'])
plt.show()

print(train['target'].value_counts(normalize=True))


In [ ]:
print("Unique keywords:", train['keyword'].nunique())
print("Sample keywords:", train['keyword'].dropna().unique()[:10])


## 🔑 Keyword Insights
We explore how informative the 'keyword' column may be.


In [ ]:
top_keywords = train['keyword'].value_counts().head(10)
sns.barplot(x=top_keywords.values, y=top_keywords.index)
plt.title("Top 10 Keywords")
plt.show()


## 🧵 Tweet Length Distribution
We check how long tweets are and whether disaster tweets tend to be longer or shorter.


In [ ]:
train['text_len'] = train['text'].apply(len)

sns.histplot(train['text_len'], bins=40, kde=True)
plt.title("Tweet Length Distribution")
plt.xlabel("Character Count")
plt.show()


In [ ]:
sns.histplot(data=train, x='text_len', hue='target', bins=40, kde=True)
plt.title("Tweet Length by Disaster Label")
plt.xlabel("Character Count")
plt.show()


## ☁️ Word Clouds
Visualizing the most common words used in disaster and non-disaster tweets.


In [ ]:
from wordcloud import WordCloud

# Combine tweets for each class
disaster_text = " ".join(train[train['target'] == 1]['text'])
nondisaster_text = " ".join(train[train['target'] == 0]['text'])

# Create word clouds
fig, axs = plt.subplots(1, 2, figsize=(16, 6))

axs[0].imshow(WordCloud(background_color='white').generate(disaster_text))
axs[0].axis('off')
axs[0].set_title('Disaster Tweets')

axs[1].imshow(WordCloud(background_color='white').generate(nondisaster_text))
axs[1].axis('off')
axs[1].set_title('Non-Disaster Tweets')

plt.show()


In [ ]:
import re
import string

def clean_text(text):
    text = text.lower()                              # Lowercase text
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'<.*?>', '', text)                # Remove HTML tags
    text = re.sub(r'\n', ' ', text)                  # Remove line breaks
    text = re.sub(r'\d+', '', text)                  # Remove numbers
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)  # Remove punctuation
    return text


In [ ]:
train['clean_text'] = train['text'].apply(clean_text)
test['clean_text'] = test['text'].apply(clean_text)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,  # Limit features for speed
    stop_words='english'  # Remove common words
)

X_train = tfidf.fit_transform(train['clean_text'])
X_test = tfidf.transform(test['clean_text'])


In [ ]:
y_train = train['target']


In [ ]:
# Tweet length
train['text_len'] = train['clean_text'].apply(len)
test['text_len'] = test['clean_text'].apply(len)

# Word count
train['word_count'] = train['clean_text'].apply(lambda x: len(x.split()))
test['word_count'] = test['clean_text'].apply(lambda x: len(x.split()))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Split training data for validation
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Train baseline Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_tr, y_tr)


In [ ]:
# Predict on validation set
y_pred = model.predict(X_val)

# Evaluate performance
print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))


In [ ]:
test_preds = model.predict(X_test)

submission = sample_submission.copy()
submission['target'] = test_preds
submission.to_csv('baseline_submission.csv', index=False)


## 📊 Baseline Model: Logistic Regression with TF-IDF

- **Model:** Logistic Regression (`sklearn.linear_model.LogisticRegression`)
- **Features:** TF-IDF (max 5000 features, English stopwords removed)
- **Train/Test Split:** 80/20
- **Validation Accuracy:** **~79.6%**

### 🔍 Class-wise Performance:

| Class | Description       | Precision | Recall | F1-Score |
|-------|-------------------|-----------|--------|----------|
| 0     | Not Disaster      | 0.78      | 0.89   | 0.83     |
| 1     | Disaster          | 0.82      | 0.67   | 0.74     |

- 🔹 The model is **strong at identifying non-disaster tweets**, but misses some real disaster tweets (lower recall).
- 🔹 Next steps: tune hyperparameters to improve class 1 recall and overall F1 score.




In [ ]:
from sklearn.model_selection import GridSearchCV

# Set up parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 5, 10, 50, 100]
}

# Grid search with 5-fold cross-validation
grid_model = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring='f1')
grid_model.fit(X_tr, y_tr)

print("Best Parameters:", grid_model.best_params_)


In [ ]:
best_model = grid_model.best_estimator_

y_pred_tuned = best_model.predict(X_val)

print("Tuned Accuracy:", accuracy_score(y_val, y_pred_tuned))
print(classification_report(y_val, y_pred_tuned))


In [ ]:
weighted_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    C=1  # Keep same regularization for fair comparison
)

weighted_model.fit(X_tr, y_tr)


In [ ]:

y_pred_weighted = weighted_model.predict(X_val)

print("Weighted Accuracy:", accuracy_score(y_val, y_pred_weighted))
print(classification_report(y_val, y_pred_weighted))


## ⚖️ Improved Recall with Class Weights

We retrained Logistic Regression using `class_weight='balanced'` to reduce bias toward the majority class.

### Results Comparison

| Metric | Baseline | Weighted |
|--------|----------|----------|
| Accuracy         | 0.796     | 0.784 |
| F1 (Class 1)     | 0.74      | 0.74  |
| Recall (Class 1) | 0.67      | 0.74  |
| Precision (Class 1) | 0.82   | 0.75  |

🧠 **Takeaway**: While accuracy dipped slightly, recall for disaster tweets improved **significantly**, making this model more sensitive to real disaster cases.

✅ Next step: explore additional models like **Naive Bayes**, or combine this approach with **engineered features** for further lift.


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# Initialize and train
nb_model = MultinomialNB()
nb_model.fit(X_tr, y_tr)


In [ ]:
y_pred_nb = nb_model.predict(X_val)

print("Naive Bayes Accuracy:", accuracy_score(y_val, y_pred_nb))
print(classification_report(y_val, y_pred_nb))


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

# Step 1: Tune TF-IDF Vectorizer
tfidf_tuned = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),         # unigrams + bigrams
    stop_words='english',
    sublinear_tf=True,          # log-scale term frequencies
    min_df=3,
    max_df=0.9
)

X_train_tuned = tfidf_tuned.fit_transform(train['clean_text'])
X_test_tuned = tfidf_tuned.transform(test['clean_text'])

y_train = train['target']

# Split for validation
X_tr, X_val, y_tr, y_val = train_test_split(X_train_tuned, y_train, test_size=0.2, random_state=42)

# Step 2: Grid Search over alpha
param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1, 5, 10]
}

grid_nb = GridSearchCV(MultinomialNB(), param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_nb.fit(X_tr, y_tr)

# Step 3: Evaluate
best_nb = grid_nb.best_estimator_

y_pred_nb_tuned = best_nb.predict(X_val)

print("Best alpha:", grid_nb.best_params_)
print("Tuned Naive Bayes Accuracy:", accuracy_score(y_val, y_pred_nb_tuned))
print(classification_report(y_val, y_pred_nb_tuned))


## 🧪 Tuning Naive Bayes with Grid Search + TF-IDF Optimization

We attempted to improve Naive Bayes performance by:
- Tuning the `alpha` smoothing parameter
- Expanding TF-IDF to include bigrams
- Using `sublinear_tf`, `min_df=3`, and `max_df=0.9`

### 🔍 Results:

| Metric | Untuned NB | Tuned NB |
|--------|------------|----------|
| Accuracy         | 0.792  | 0.790 |
| F1 (Disaster)    | 0.74   | 0.73  |
| Recall (Disaster)| 0.68   | 0.65  |
| Precision        | 0.80   | 0.82  |

🎯 Conclusion: Precision improved slightly, but recall decreased. Best `alpha` was 1.0 — same as default. Tuning did not significantly improve performance.


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("CUDA Available:", torch.cuda.is_available())


In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split


In [ ]:
# We assume this exists from earlier:
# train['clean_text'] and train['target']

df = train[['clean_text', 'target']].copy()


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)


In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))


In [ ]:
train_ds[0]


In [ ]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

def tokenize_function(example):
    return tokenizer(
        example["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)


In [ ]:
from transformers import BertForSequenceClassification

# 1) Load BERT for binary classification
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

# 2) Move to GPU
model = model.to('cuda')
print("Model device:", next(model.parameters()).device)  # should print cuda:0


In [ ]:
# Prepare 1 step of real training to check GPU usage
from torch.utils.data import DataLoader
import torch

# Make sure your train_ds/val_ds are tokenized and have column 'labels'
tiny_train = train_ds.shuffle(seed=0).select(range(16)).rename_column("target","labels")
tiny_train.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
train_loader = DataLoader(tiny_train, batch_size=8)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)

model.train()
for batch in train_loader:
    input_ids = batch["input_ids"].cuda()
    attn_mask = batch["attention_mask"].cuda()
    labels    = batch["labels"].cuda()
    out = model(input_ids, attention_mask=attn_mask, labels=labels)
    print("Loss:", out.loss.item())
    out.loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    break


In [ ]:
# 1) Make sure your model is on CUDA
model = model.to("cuda")
print("Model device:", next(model.parameters()).device)  # should say cuda:0

# 2) Prepare a tiny DataLoader from a handful of examples
from torch.utils.data import DataLoader
import torch

# pick 16 training items and 4 validation items
tiny_train = train_ds.shuffle(seed=0).select(range(16))
tiny_val   = val_ds.shuffle(seed=0).select(range(4))

# rename 'target'→'labels' if needed
if "target" in tiny_train.column_names:
    tiny_train = tiny_train.rename_column("target","labels")
    tiny_val   = tiny_val.rename_column("target","labels")

# convert to torch format
tiny_train.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
tiny_val  .set_format(type="torch", columns=["input_ids","attention_mask","labels"])

train_loader = DataLoader(tiny_train, batch_size=8)
val_loader   = DataLoader(tiny_val,   batch_size=4)

# 3) One forward+backward pass
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)

model.train()
for batch in train_loader:
    input_ids = batch["input_ids"].cuda()
    attn_mask = batch["attention_mask"].cuda()
    labels    = batch["labels"].cuda()
    out = model(input_ids, attention_mask=attn_mask, labels=labels)
    print("Loss:", out.loss.item())
    out.loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    break  # only one step

# 4) Confirm that the GPU bar lights up during that step!


In [ ]:
train_ds[0]


In [ ]:
train_ds = train_ds.rename_column('target','labels')
val_ds   = val_ds.rename_column('target','labels')

train_ds.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

val_ds.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)


In [ ]:
train_ds[0]


In [ ]:


from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2  # Binary classification: 0 = not disaster, 1 = disaster
)


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',               # where to save the model
    evaluation_strategy="epoch",          # evaluate after each epoch
    save_strategy="epoch",                # save model after each epoch
    num_train_epochs=1,                   # you can increase later
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,                     # helps stabilize training early
    weight_decay=0.01,                    # prevents overfitting
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',

    fp16=True,
    report_to=[]
)


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


In [ ]:
print(train_ds.column_names, len(train_ds))


In [ ]:
# 1) Ensure model is the BERT model and on GPU
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
).to('cuda')
print("Model device:", next(model.parameters()).device)  # should print cuda:0

# 2) Prepare a tiny DataLoader for one forward+backward pass
from torch.utils.data import DataLoader
import torch

# Rename if needed
if 'target' in train_ds.column_names:
    train_ds = train_ds.rename_column('target', 'labels')
train_ds.set_format(type='torch', columns=['input_ids','attention_mask','labels'])

tiny = train_ds.shuffle(seed=0).select(range(16))  # take 16 examples
loader = DataLoader(tiny, batch_size=8)

# 3) Run one training step
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)

model.train()
for batch in loader:
    input_ids = batch['input_ids'].cuda()
    attn_mask = batch['attention_mask'].cuda()
    labels    = batch['labels'].cuda()

    out = model(input_ids, attention_mask=attn_mask, labels=labels)
    print("Loss:", out.loss.item())

    out.loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    break  # only one step


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)



In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Model device:", next(model.parameters()).device)


In [ ]:
import torch, time

# allocate two large matrices on the GPU
a = torch.randn(8000, 8000, device='cuda')
b = torch.randn(8000, 8000, device='cuda')

# make sure all prior CUDA calls are done
torch.cuda.synchronize()

# time a loop of 20 multiplies
t0 = time.time()
for _ in range(20):
    c = a @ b
torch.cuda.synchronize()
print(f"Elapsed {time.time() - t0:.2f}s")


In [ ]:
trainer.train()


In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# Predict on validation
preds_output = trainer.predict(val_ds)
preds = np.argmax(preds_output.predictions, axis=1)
labels = preds_output.label_ids

print(classification_report(labels, preds, target_names=["Not Disaster","Disaster"]))


In [ ]:
# Tokenize the test set (ensure you have test['clean_text'])
test_enc = tokenizer(
    test['clean_text'].tolist(),
    truncation=True,
    padding='max_length',
    max_length=64,           # match your training setting
    return_tensors='pt'
)

model.eval()
with torch.no_grad():
    outputs = model(
        input_ids=test_enc['input_ids'].cuda(),
        attention_mask=test_enc['attention_mask'].cuda()
    )
preds = torch.argmax(outputs.logits, axis=1).cpu().numpy()

# Build submission
submission = sample_submission.copy()
submission['target'] = preds
submission.to_csv("submission.csv", index=False)
print("Saved to submission.csv")
